# API'S Extraction and standardizing process:# Hotel Data Extraction & Preparation

## Overview

This notebook is part of a larger data analysis project focused on understanding hotel pricing dynamics.

The objective of this notebook is to **extract, structure, and clean hotel data** from external APIs, creating a reliable base dataset for further analysis.

## What this notebook does

- Connects to external APIs (Google Hotel Data via RapidAPI)
- Retrieves city-level data and extracts city codes (IATA)
- Collects hotel data for selected cities (Paris and Rome)
- Transforms raw JSON responses into structured DataFrames
- Cleans and standardizes the dataset
- Removes duplicates and validates data quality
- Exports a processed dataset ready for analysis

## Output

The final result of this notebook is a cleaned dataset:

`data/processed/hotels_paris_rome_base_clean.csv`

This dataset will be used in future steps of the project, where it will be combined with additional data sources such as events and weather.

## Why this matters

Having a clean and well-structured dataset is essential before performing any meaningful analysis.  
This notebook focuses on building a **solid data foundation layer**, ensuring data quality and consistency.

## Notes

- API rate limits may apply depending on the subscription plan
- Environment variables are used to securely manage API credentials

## Setup project path

This cell ensures the notebook can import modules from the project.

- `Path().resolve().parent` gets the project’s root directory  
- `sys.path.append(...)` adds it to Python’s module search path  

This allows importing files from the project (e.g., `from src.config import ...`) without errors.

In [61]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

## Load and verify API configuration

This cell reloads the `config` file and imports the main variables needed to connect to the API.

It is used to make sure the notebook is working with the latest configuration values, especially after updating the `.env` file or the `config.py` file.

Then, it prints a quick check to confirm that:

- the RapidAPI key is not empty
- the API host has been loaded correctly

This is a simple validation step before making requests to the hotel API.

In [62]:
import importlib
import src.config

importlib.reload(src.config)

from src.config import RAPIDAPI_KEY, GOOGLE_HOTEL_RAPIDAPI_HOST, is_non_empty

print("RAPIDAPI_KEY loaded:", is_non_empty(RAPIDAPI_KEY))
print("GOOGLE_HOTEL_RAPIDAPI_HOST:", GOOGLE_HOTEL_RAPIDAPI_HOST)

RAPIDAPI_KEY loaded: True
GOOGLE_HOTEL_RAPIDAPI_HOST: google-hotel-data-scaper.p.rapidapi.com


## Test the city search endpoint and inspect the response safely

This cell sends a request to the city search endpoint of the Google Hotel API using **Paris** as the search keyword.

It defines the API URL, the required headers, and the query parameter, then performs a `GET` request to retrieve the response.

After that, it prints some basic information about the request result, such as:

- the **status code**
- the **content type**
- the **keys** returned in the JSON response
- a short **preview** of the response content

Before converting the response into a DataFrame, the code checks whether the `"data"` field exists and whether it contains a list.

If the response is valid, the `"data"` field is normalized into a pandas DataFrame and the code displays:

- the **shape** of the DataFrame
- the **first rows**
- the **column names**

If the API does not return a valid `"data"` field, the code prints the full response instead of failing.

This makes the workflow safer and helps detect API errors or unexpected response formats.

In [ ]:
import requests
import pandas as pd

from src.config import RAPIDAPI_KEY, GOOGLE_HOTEL_RAPIDAPI_HOST

url = "https://google-hotel-data-scaper.p.rapidapi.com/hotel/reference-data-locations-cities.php"

headers = {
    "Content-Type": "application/json",
    "x-rapidapi-host": GOOGLE_HOTEL_RAPIDAPI_HOST,
    "x-rapidapi-key": RAPIDAPI_KEY
}

params = {"keyword": "Paris"}

response = requests.get(url, headers=headers, params=params, timeout=30)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

data = response.json()

print("Response keys:", data.keys())
print("Response preview:", str(data)[:1000])

if "data" in data and isinstance(data["data"], list):
    cities_df = pd.json_normalize(data["data"])
    print("Shape:", cities_df.shape)
    display(cities_df.head())
    print(cities_df.columns.tolist())
else:
    print("No 'data' field found in the API response.")
    print("Full response:", data)

Status code: 429
Content-Type: application/json
Response preview:
{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/kingmakerapi\/api\/google-hotel-data-scaper"}


## Create a structured DataFrame from the API response

This cell takes the list of cities stored in the `"data"` field of the API response and converts it into a pandas DataFrame.

`pd.json_normalize()` is used to flatten the JSON structure into a tabular format, which makes the information easier to read and work with.

At the end, the code shows:

- the **shape** of the DataFrame
- the **first rows**
- the **list of columns**

This is a basic inspection step to understand the structure of the returned city data before filtering or analyzing it further.

In [ ]:
paris_city_df = cities_df[
    (cities_df["name"].str.lower() == "paris") &
    (cities_df["address.countryCode"] == "FR")
].copy()

display(paris_city_df)

,type,subType,name,iataCode,address.countryCode,address.stateCode,geoCode.latitude,geoCode.longitude
0,location,city,Paris,PAR,FR,FR-75,48.85341,2.3488


## Extract the city code for Paris

This cell selects the `iataCode` value from the first row of the filtered DataFrame for Paris.

`.iloc[0]` is used to access the first matching result.

The extracted city code is stored in the variable `paris_city_code` and printed, so it can be used in the next API requests, such as searching for hotels or offers in Paris.

In [ ]:
paris_city_code = paris_city_df["iataCode"].iloc[0]
print("Paris city code:", paris_city_code)

Paris city code: PAR


## Test the city search endpoint for Rome

This cell sends a request to the same city reference endpoint, but this time using **Rome** as the search keyword.

It sets the API URL, headers, and query parameters, then performs a `GET` request to retrieve the matching city data.

After that, it prints:

- the **status code** to verify whether the request was successful
- the **content type** to confirm the response format
- the **type of the parsed response**
- the **top-level keys** in the returned JSON

This step is useful to check whether the API returns the expected structure for Rome before converting the response into a DataFrame.

In [ ]:
import requests
import pandas as pd

from src.config import RAPIDAPI_KEY, GOOGLE_HOTEL_RAPIDAPI_HOST

url = "https://google-hotel-data-scaper.p.rapidapi.com/hotel/reference-data-locations-cities.php"

headers = {
    "Content-Type": "application/json",
    "x-rapidapi-host": GOOGLE_HOTEL_RAPIDAPI_HOST,
    "x-rapidapi-key": RAPIDAPI_KEY
}

params = {
    "keyword": "Rome"
}

response = requests.get(url, headers=headers, params=params, timeout=30)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

data = response.json()

print(type(data))
print(data.keys())

Status code: 200
Content-Type: application/json
<class 'dict'>
dict_keys(['meta', 'data'])


## Convert Rome city data into a DataFrame

This cell takes the `"data"` field from the API response for Rome and converts it into a pandas DataFrame using `pd.json_normalize()`.

This makes the JSON response easier to explore in a tabular format.

Then, it displays:

- the **shape** of the DataFrame to see its size
- the **column names** to understand the available fields
- the **first rows** to preview the returned city records

This step helps inspect the Rome results before selecting the correct city entry or extracting its code.

In [ ]:
df_rome_cities = pd.json_normalize(data["data"])

print("Shape:", df_rome_cities.shape)
print(df_rome_cities.columns.tolist())

df_rome_cities.head()

Shape: (86, 9)
['type', 'subType', 'name', 'iataCode', 'address.countryCode', 'address.stateCode', 'geoCode.latitude', 'geoCode.longitude', 'geoCode']


,type,subType,name,iataCode,address.countryCode,address.stateCode,geoCode.latitude,geoCode.longitude,geoCode
0,location,city,Rome,ROM,IT,IT-ZZZ,41.89193,12.51133,NaN
1,location,city,Romeoville,LOT,US,US-IL,41.64753,-88.08951,NaN
2,location,city,Font-Romeu-Odeillo-Via,QZF,FR,FR-66,42.50552,2.04011,NaN
3,location,city,Rome,REO,US,US-OR,NaN,NaN,[]
4,location,city,Rome,RME,US,US-NY,43.21285,-75.45573,NaN


## Identify the correct Rome record and extract its city code

This section first displays a few key columns to inspect the available city results for Rome.

Then, it filters the DataFrame to keep only the row where:

- the city name is **Rome**
- the country code is **IT** (Italy)

This helps make sure the correct city is selected in case the API returns multiple matches.

After filtering, the code extracts the `iataCode` from the first matching row and stores it in `rome_city_code`.

Finally, both city codes are defined as constants:

- `PARIS_CITY_CODE = "PAR"`
- `ROME_CITY_CODE = "ROM"`

These constants are useful for later steps, such as requesting hotel data or offers for each city.

In [ ]:
df_rome_cities[["name", "iataCode", "address.countryCode", "address.stateCode"]]

,name,iataCode,address.countryCode,address.stateCode
0,Rome,ROM,IT,IT-ZZZ
1,Romeoville,LOT,US,US-IL
2,Font-Romeu-Odeillo-Via,QZF,FR,FR-66
3,Rome,REO,US,US-OR
4,Rome,RME,US,US-NY
...,...,...,...,...
81,Rome City,NaN,US,US-IN
82,Rome,NaN,US,US-ME
83,Jerome,NaN,US,US-PA
84,Rome,NaN,US,US-WI


In [ ]:
rome_row = df_rome_cities[
    (df_rome_cities["name"].str.lower() == "rome") &
    (df_rome_cities["address.countryCode"] == "IT")
].copy()

rome_row

,type,subType,name,iataCode,address.countryCode,address.stateCode,geoCode.latitude,geoCode.longitude,geoCode
0,location,city,Rome,ROM,IT,IT-ZZZ,41.89193,12.51133,NaN


In [ ]:
rome_city_code = rome_row.iloc[0]["iataCode"]
print("Rome city code:", rome_city_code)

Rome city code: ROM


In [ ]:
PARIS_CITY_CODE = "PAR"
ROME_CITY_CODE = "ROM"

## Define a function to get hotels by city

This cell creates a reusable function called `get_hotels_by_city()` that retrieves hotel reference data for a given city.

The function:

- receives a city code as input
- defines the hotel-by-city endpoint
- sets the required API headers
- sends the city code as a query parameter
- makes a `GET` request to the API

`response.raise_for_status()` is used to stop execution if the request fails, which helps catch API errors early.

Finally, the function returns the response in JSON format.

This makes it easier to request hotel data for different cities without repeating the same code each time.

In [ ]:
import requests
import pandas as pd

from src.config import RAPIDAPI_KEY, GOOGLE_HOTEL_RAPIDAPI_HOST


def get_hotels_by_city(city_code: str) -> dict:
    url = "https://google-hotel-data-scaper.p.rapidapi.com/hotel/reference-data-locations-hotels-by-city.php"

    headers = {
        "Content-Type": "application/json",
        "x-rapidapi-host": GOOGLE_HOTEL_RAPIDAPI_HOST,
        "x-rapidapi-key": RAPIDAPI_KEY
    }

    params = {
        "cityCode": city_code
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    return response.json()

## Retrieve and structure hotel data for Paris and Rome

This cell uses the `get_hotels_by_city()` function to request hotel data for both **Paris** and **Rome**.

First, it stores the raw JSON responses in two variables:

- `paris_hotels_raw`
- `rome_hotels_raw`

Then, it prints the type of each object and their top-level keys to verify that both API responses were loaded correctly.

After that, the `"data"` field from each response is converted into a pandas DataFrame using `pd.json_normalize()`. This transforms the hotel data into a structured tabular format.

Finally, the code prints the shape of each DataFrame and displays the first rows of both:

- `paris_hotels_df`
- `rome_hotels_df`

This step allows us to compare how many hotel records were returned for each city and inspect the structure of the hotel-level data before combining or analyzing it further.

In [ ]:
paris_hotels_raw = get_hotels_by_city(PARIS_CITY_CODE)
rome_hotels_raw = get_hotels_by_city(ROME_CITY_CODE)

print(type(paris_hotels_raw))
print(type(rome_hotels_raw))
print(paris_hotels_raw.keys())
print(rome_hotels_raw.keys())

<class 'dict'>
<class 'dict'>
dict_keys(['data', 'meta'])
dict_keys(['data', 'meta'])


In [ ]:
paris_hotels_df = pd.json_normalize(paris_hotels_raw["data"])
rome_hotels_df = pd.json_normalize(rome_hotels_raw["data"])

print("Paris shape:", paris_hotels_df.shape)
print("Rome shape:", rome_hotels_df.shape)

display(paris_hotels_df.head())
display(rome_hotels_df.head())

Paris shape: (1679, 14)
Rome shape: (465, 14)


,chainCode,iataCode,dupeId,name,hotelId,lastUpdate,geoCode.latitude,geoCode.longitude,address.countryCode,address.postalCode,address.cityName,address.lines,retailing.sponsorship.isSponsored,masterChainCode
0,IW,PAR,700041562,MAISON ALBAR-LE CHAMPS-ELYSEES,IWPARMMP,2025-12-22T11:57:46,48.87541,2.29453,FR,75017,PARIS,[3 AVENUE MAC MAHON],True,NaN
1,BN,PAR,700031222,OCCIDENTAL PARIS LEVALLOIS,BNPAR058,2026-04-09T11:21:08,48.89818,2.28087,FR,92300,PARIS,"[8 PLACE GEORGES POMPIDOU, LEVALLOIS PERRET]",True,NaN
2,WA,PAR,700027367,WLDRF ASTORIA VERSAILLES TRIANON PALACE,WAPARTRI,2026-04-11T16:24:20,48.81067,2.12022,FR,78000,VERSAILLES - PARIS,[1 BOULEVARD DE LA REINE],True,EH
3,PN,PAR,700222607,THE PENINSULA PARIS,PNPARTPP,2026-03-06T08:51:54,48.87073,2.29340,FR,75116,PARIS,[19 AVENUE KLEBER],True,NaN
4,HL,PAR,700024440,HILTON PARIS DEFENSE,HLPAR100,2026-04-14T08:25:09,48.89305,2.23857,FR,92053,LA DEFENSE,"[2 PLACE DE LA DEFENSE, CNIT-B]",True,EH


,chainCode,iataCode,dupeId,name,hotelId,masterChainCode,lastUpdate,geoCode.latitude,geoCode.longitude,address.countryCode,address.postalCode,address.cityName,address.lines,retailing.sponsorship.isSponsored
0,NH,ROM,700008832,NHOW ROMA,NHROMJRM,MW,2026-02-06T17:20:53,41.91017,12.49022,IT,00198,Roma,"[CORSO D'ITALIA, 1]",True
1,HY,ROM,700021356,HYATT REGENCY ROME CENTRAL,HYROMRT7,LH,2025-10-21T08:12:06,41.89673,12.50597,IT,00185,ROME,[VIA FILIPPO TURATI 171],True
2,HL,ROM,700965286,HILTON ROME EUR LA LAMA,HLROM533,EH,2026-03-29T12:37:44,41.83050,12.47134,IT,00144,ROME,[VIALE EUROPA SNC],True
3,PK,ROM,700059941,ARTOTEL ROME PIAZZA SALLUSTIO,PKROM566,CW,2026-03-21T21:13:23,41.90856,12.49719,IT,00187,ROME,"[PIAZZA SALLUSTIO, 18]",True
4,BN,ROM,700026236,HOTEL MIDAS ROMA MEMBER OF BHG,BNROMMID,NaN,2026-02-18T14:27:36,41.88824,12.40241,IT,00165,ROME,[VIA RAFFAELLO SARDIELLO 22],True


## Combine hotel data from Paris and Rome

This cell adds a new column called `search_city` to each hotel DataFrame, so it is clear which city each hotel record comes from after merging the datasets.

Then, both DataFrames are combined into a single DataFrame called `hotels_df_raw` using `pd.concat()`.

After merging, the code performs a basic inspection by showing:

- the **shape** of the combined DataFrame
- the **first rows**
- the **list of columns**
- the output of `.info()` to check data types and missing values

Finally, the combined raw dataset is saved as a CSV file in the `data/raw/` folder.

This step creates one unified hotel dataset for both cities, ready for cleaning and further analysis.

In [ ]:
paris_hotels_df["search_city"] = "Paris"
rome_hotels_df["search_city"] = "Rome"

In [ ]:
hotels_df_raw = pd.concat([paris_hotels_df, rome_hotels_df], ignore_index=True)

print("Combined shape:", hotels_df_raw.shape)
display(hotels_df_raw.head())

Combined shape: (2144, 15)


,chainCode,iataCode,dupeId,name,hotelId,lastUpdate,geoCode.latitude,geoCode.longitude,address.countryCode,address.postalCode,address.cityName,address.lines,retailing.sponsorship.isSponsored,masterChainCode,search_city
0,IW,PAR,700041562,MAISON ALBAR-LE CHAMPS-ELYSEES,IWPARMMP,2025-12-22T11:57:46,48.87541,2.29453,FR,75017,PARIS,[3 AVENUE MAC MAHON],True,NaN,Paris
1,BN,PAR,700031222,OCCIDENTAL PARIS LEVALLOIS,BNPAR058,2026-04-09T11:21:08,48.89818,2.28087,FR,92300,PARIS,"[8 PLACE GEORGES POMPIDOU, LEVALLOIS PERRET]",True,NaN,Paris
2,WA,PAR,700027367,WLDRF ASTORIA VERSAILLES TRIANON PALACE,WAPARTRI,2026-04-11T16:24:20,48.81067,2.12022,FR,78000,VERSAILLES - PARIS,[1 BOULEVARD DE LA REINE],True,EH,Paris
3,PN,PAR,700222607,THE PENINSULA PARIS,PNPARTPP,2026-03-06T08:51:54,48.87073,2.29340,FR,75116,PARIS,[19 AVENUE KLEBER],True,NaN,Paris
4,HL,PAR,700024440,HILTON PARIS DEFENSE,HLPAR100,2026-04-14T08:25:09,48.89305,2.23857,FR,92053,LA DEFENSE,"[2 PLACE DE LA DEFENSE, CNIT-B]",True,EH,Paris


In [ ]:
print(hotels_df_raw.columns.tolist())

['chainCode', 'iataCode', 'dupeId', 'name', 'hotelId', 'lastUpdate', 'geoCode.latitude', 'geoCode.longitude', 'address.countryCode', 'address.postalCode', 'address.cityName', 'address.lines', 'retailing.sponsorship.isSponsored', 'masterChainCode', 'search_city']


In [ ]:
hotels_df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2144 entries, 0 to 2143
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   chainCode                          2144 non-null   object 
 1   iataCode                           2144 non-null   object 
 2   dupeId                             2144 non-null   int64  
 3   name                               2144 non-null   object 
 4   hotelId                            2144 non-null   object 
 5   lastUpdate                         2144 non-null   object 
 6   geoCode.latitude                   2144 non-null   float64
 7   geoCode.longitude                  2144 non-null   float64
 8   address.countryCode                2144 non-null   object 
 9   address.postalCode                 2143 non-null   object 
 10  address.cityName                   2144 non-null   object 
 11  address.lines                      2144 non-null   objec

In [ ]:
hotels_df_raw.to_csv(project_root / "data" / "raw" / "hotels_paris_rome_raw.csv", index=False)

## Clean and standardize the hotel dataset

This section creates a clean working copy of the combined hotel dataset and prepares it for analysis.

First, the columns are renamed from the original API format to a cleaner **snake_case** format, which makes them easier to read and use in Python.

Then, several data types are corrected:

- `latitude` and `longitude` are converted to numeric values
- `last_update` is converted to datetime format
- `is_sponsored` is filled with `False` when missing and converted to boolean

Next, the main text columns are standardized by converting them to strings and removing extra spaces with `.str.strip()`. This helps avoid inconsistencies in identifiers or text fields.

The code then checks the uniqueness of `hotel_id` by printing:

- the number of duplicated hotel IDs
- the number of unique hotel IDs

After that, duplicate hotel records are removed based on `hotel_id`, keeping only one row per hotel.

Finally, the code:

- prints the new shape of the dataset after deduplication
- shows the number of missing values per column
- saves the cleaned base dataset as a CSV file in the `data/processed/` folder

This step is essential because it turns the raw hotel data into a more reliable and analysis-ready dataset.

In [ ]:
hotels_df = hotels_df_raw.copy()

In [ ]:
hotels_df = hotels_df.rename(columns={
    "chainCode": "chain_code",
    "iataCode": "iata_code",
    "dupeId": "dupe_id",
    "name": "hotel_name",
    "hotelId": "hotel_id",
    "lastUpdate": "last_update",
    "geoCode.latitude": "latitude",
    "geoCode.longitude": "longitude",
    "address.countryCode": "country_code",
    "address.postalCode": "postal_code",
    "address.cityName": "city_name",
    "address.lines": "address_lines",
    "retailing.sponsorship.isSponsored": "is_sponsored",
    "masterChainCode": "master_chain_code"
})

In [ ]:
hotels_df["latitude"] = pd.to_numeric(hotels_df["latitude"], errors="coerce")
hotels_df["longitude"] = pd.to_numeric(hotels_df["longitude"], errors="coerce")
hotels_df["last_update"] = pd.to_datetime(hotels_df["last_update"], errors="coerce")
hotels_df["is_sponsored"] = hotels_df["is_sponsored"].fillna(False).astype(bool)

C:\Users\johan\AppData\Local\Temp\ipykernel_20348\3102766891.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hotels_df["is_sponsored"] = hotels_df["is_sponsored"].fillna(False).astype(bool)


In [ ]:
text_cols = [
    "chain_code", "iata_code", "dupe_id", "hotel_name", "hotel_id",
    "country_code", "postal_code", "city_name", "master_chain_code", "search_city"
]

for col in text_cols:
    hotels_df[col] = hotels_df[col].astype(str).str.strip()

In [ ]:
print("Duplicated hotel_id:", hotels_df["hotel_id"].duplicated().sum())
print("Unique hotel_id:", hotels_df["hotel_id"].nunique())

Duplicated hotel_id: 0
Unique hotel_id: 2144


In [ ]:
hotels_df = hotels_df.drop_duplicates(subset="hotel_id").reset_index(drop=True)

print("Shape after dedup:", hotels_df.shape)

Shape after dedup: (2144, 15)


In [ ]:
print(hotels_df.isna().sum().sort_values(ascending=False))

chain_code           0
iata_code            0
dupe_id              0
hotel_name           0
hotel_id             0
last_update          0
latitude             0
longitude            0
country_code         0
postal_code          0
city_name            0
address_lines        0
is_sponsored         0
master_chain_code    0
search_city          0
dtype: int64


In [ ]:
hotels_df.to_csv(project_root / "data" / "processed" / "hotels_paris_rome_base_clean.csv", index=False)